# Spark SQL: Análisis Relacional Complejo

En este notebook, instanciamos un recorte de nuestro Catálogo Relacional (DML) en DataFrames de Spark SQL.

El objetivo es aprovechar el motor distribuido para resolver consultas complejas de Trazabilidad, Compliance, Operaciones y Calidad, demostrando cómo la arquitectura respondería ante escenarios analíticos de alto costo computacional.

In [ ]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('TP_SparkSQL_Gobierno').getOrCreate()
print('Spark Session iniciada.')

# --- CREACION DE DATAFRAMES (Mapeo relacional del SQL del TP1) ---

df_fuente = spark.createDataFrame([
    (1, 'DWH Finanzas Oracle', 'Relacional'), (2, 'DWH Ventas SQL Server', 'Relacional'),
    (3, 'Data Lake Operaciones S3', 'No Relacional')
], ['id_fuente', 'nombre', 'tipo_fuente'])
df_fuente.createOrReplaceTempView('fuente_de_datos')

df_activo = spark.createDataFrame([
    (1, 'Reporte Ingreso Mensual', 'Objeto'), (36, 'Tablero Retencion y Churn', 'Objeto'),
    (46, 'ETL Ingesta Ventas', 'Objeto'), (61, 'Tabla Maestra Clientes', 'Datos Estructurado'),
    (83, 'Modelo Predictivo Churn', 'Modelo')
], ['id_activo', 'nombre', 'tipo_activo'])
df_activo.createOrReplaceTempView('activo')

df_linaje = spark.createDataFrame([
    (46, 61), (61, 83), (61, 1), (61, 36)
], ['id_activo_origen', 'id_activo_destino'])
df_linaje.createOrReplaceTempView('linaje_datos')

df_fio = spark.createDataFrame([
    (1, 1), (1, 36), (2, 46)
], ['id_fuente', 'id_activo'])
df_fio.createOrReplaceTempView('fuente_impacta_objeto')

df_equipo = spark.createDataFrame([
    (1, 'Contabilidad General', 1), (8, 'Datos e Inteligencia', 2), (10, 'Analitica de Clientes', 3)
], ['id_equipo', 'nombre', 'id_departamento'])
df_equipo.createOrReplaceTempView('equipo')

df_eto = spark.createDataFrame([
    (1, 1), (36, 1), (46, 8), (61, 10), (83, 10)
], ['id_activo', 'id_equipo'])
df_eto.createOrReplaceTempView('equipo_trabaja_objeto')

df_operacion = spark.createDataFrame([
    (1, 'Cierre contable mensual', 1), (6, 'Despliegue pipelines ETL', 2), (9, 'Segmentacion de audiencias', 3)
], ['id_operacion', 'descripcion', 'id_departamento'])
df_operacion.createOrReplaceTempView('operacion')

df_ova = spark.createDataFrame([
    (1, 1), (6, 46), (9, 83)
], ['id_operacion', 'id_activo'])
df_ova.createOrReplaceTempView('operacion_vinculada_activo')

df_problema = spark.createDataFrame([
    (1, 'Datos duplicados en clientes', 'Resuelto', 2, 61),
    (10, 'Timeout en query', 'No Resuelto', 1, 36)
], ['id_problema', 'descripcion', 'estado', 'id_fuente', 'id_activo'])
df_problema.createOrReplaceTempView('problema')

df_sujeto = spark.createDataFrame([
    (1001, 'Miembro UO'), (1010, 'Miembro UO')
], ['id_sujeto', 'tipo_sujeto'])
df_sujeto.createOrReplaceTempView('sujeto')

df_accesos = spark.createDataFrame([
    (1001, 1, 2, 4), (1001, 2, 2, 4), (1010, 1, 3, 2)
], ['id_sujeto', 'id_fuente', 'id_rol', 'id_perfil'])
df_accesos.createOrReplaceTempView('acceso_fuente_datos')

df_herramienta = spark.createDataFrame([
    (2, 'Power BI'), (6, 'AWS Glue'), (23, 'Databricks Lakehouse')
], ['id_herramienta', 'nombre'])
df_herramienta.createOrReplaceTempView('herramienta')

df_hga = spark.createDataFrame([
    (1, 2), (46, 6), (83, 23)
], ['id_activo', 'id_herramienta'])
df_hga.createOrReplaceTempView('herramienta_gestiona_activo')

print('Vistas temporales creadas. Todo el modelo relacional instanciado.')

### Caso 1: Trazabilidad y Linaje (Fuente Física -> Activo Lógico -> Equipo)

En un ecosistema de datos, si una base de datos física (origen) sufre una caída, es vital saber a quién impacta. Esta consulta permite trazar el camino inverso: si falla la Fuente Física, sabemos exactamente qué Activos de Datos se corrompen y a qué Equipo responsable debemos notificar de forma proactiva.

In [ ]:
query_1 = '''
    SELECT f.nombre AS Fuente_Origen,
           a.nombre AS Objeto,
           e.nombre AS Equipo_Responsable
    FROM fuente_de_datos f
    JOIN fuente_impacta_objeto fio ON f.id_fuente = fio.id_fuente
    JOIN activo a ON fio.id_activo = a.id_activo
    JOIN equipo_trabaja_objeto eto ON a.id_activo = eto.id_activo
    JOIN equipo e ON eto.id_equipo = e.id_equipo
'''
resultado_1 = spark.sql(query_1)

# Mostrar los resultados como tablas
resultado_1.show(truncate=False)

**Análisis:** Logramos mapear la cadena de impacto completa. Por ejemplo, ante una caída del "DWH Finanzas Oracle", sabemos automáticamente que el área de "Contabilidad General" perderá acceso a su Reporte de Ingreso y Tablero de Churn, permitiendo una comunicación proactiva de incidentes.

### Caso 2: Impacto de Incidentes en Operaciones de Negocio

Los problemas técnicos (ej. un timeout en una base de datos) no significan nada para los ejecutivos si no se traducen a impacto operativo. Esta consulta cruza los incidentes de calidad de datos con los procesos de la empresa, calculando el "riesgo operativo" diario.

In [ ]:
query_2 = '''
    SELECT p.descripcion AS Falla_Reportada,
           a.nombre AS Activo_Afectado,
           o.descripcion AS Operacion_Impactada
    FROM problema p
    JOIN activo a ON p.id_activo = a.id_activo
    JOIN operacion_vinculada_activo ova ON a.id_activo = ova.id_activo
    JOIN operacion o ON ova.id_operacion = o.id_operacion
    WHERE p.estado = 'No Resuelto'
'''
resultado_2 = spark.sql(query_2)

# Mostrar los resultados como tablas
resultado_2.show(truncate=False)

**Análisis:** El motor arrojó un conjunto vacío. Lejos de ser un error, este es un insight sumamente positivo: demuestra que ninguna de las fallas técnicas actuales (Deuda Técnica No Resuelta) está bloqueando Operaciones Críticas de Negocio en este instante, manteniendo el riesgo aislado.

### Caso 3: Mapa de Grafo Lógico (Linaje Activo a Activo)

Es el núcleo del Compliance y el Gobierno de Datos. Demuestra de manera documentada cómo un dato nace, se transforma y se consume. Sirve para que un Analista de Datos entienda de qué tabla maestra se está alimentando su Tablero o su Modelo Predictivo.

In [ ]:
query_3 = '''
    SELECT origen.nombre AS Activo_Padre,
           destino.nombre AS Objeto_Hijo,
           destino.tipo_activo
    FROM linaje_datos l
    JOIN activo origen ON l.id_activo_origen = origen.id_activo
    JOIN activo destino ON l.id_activo_destino = destino.id_activo
'''
resultado_3 = spark.sql(query_3)

# Mostrar los resultados como tablas
resultado_3.show(truncate=False)

**Análisis:** Identificamos que la "Tabla Maestra Clientes" es un nodo crítico, ya que su alteración impactaría simultáneamente en capas de visualización (Reportes/Tableros) y en algoritmos de Machine Learning (Modelo Predictivo).

### Caso 4: Infraestructura vs Activos (Qué herramientas soportan qué objetos)

Permite realizar un análisis de dependencias tecnológicas y control de costos corporativos. Ayuda a los Arquitectos de Datos a planificar migraciones o estimar costos de licenciamiento (Cloud/Software), viendo claramente qué herramientas soportan a qué objetos (tableros, ETLs, modelos).

In [ ]:
query_4 = '''
    SELECT h.nombre AS Herramienta_Proveedor,
           a.nombre AS Activo_Implementado,
           a.tipo_activo
    FROM herramienta h
    JOIN herramienta_gestiona_activo hga ON h.id_herramienta = hga.id_herramienta
    JOIN activo a ON hga.id_activo = a.id_activo
'''
resultado_4 = spark.sql(query_4)

# Mostrar los resultados como tablas
resultado_4.show(truncate=False)

**Análisis:** Conseguimos un inventario vivo de licenciamiento. Validamos la integración de tecnologías Cloud (AWS Glue) para ingesta, almacenamiento Lakehouse (Databricks) para modelos y software propietario (Power BI) para consumo estructurado.

### Caso 5: Auditoría de Seguridad Perimetral (Sujetos y Fuentes Autorizadas)

Soporta las políticas corporativas de Data Security. Genera una matriz de accesos automatizada que permite a los auditores de seguridad verificar quién (empleado o externo) puede acceder a qué fuente física de datos y con qué nivel de privilegios (perfil).

In [ ]:
query_5 = '''
    SELECT s.id_sujeto,
           s.tipo_sujeto,
           f.nombre AS Fuente_Autorizada,
           afd.id_perfil AS Nivel_Privilegio
    FROM sujeto s
    JOIN acceso_fuente_datos afd ON s.id_sujeto = afd.id_sujeto
    JOIN fuente_de_datos f ON afd.id_fuente = f.id_fuente
    ORDER BY s.id_sujeto
'''
resultado_5 = spark.sql(query_5)

# Mostrar los resultados como tablas
resultado_5.show(truncate=False)

**Análisis:** La matriz perimetral alerta al Oficial de Seguridad (CISO) sobre el Sujeto 1001, quien concentra privilegios de Nivel 4 en múltiples bases de datos críticas simultáneamente. Este acceso, en un entorno real, debe ser revalidado periódicamente.

### Caso 6: Carga de Trabajo Analítica por Equipo

Provee un Indicador Clave de Rendimiento (KPI) para la gerencia operativa. Permite visualizar qué departamentos o equipos están gestionando y sosteniendo la mayor cantidad de activos analíticos, ayudando a balancear la carga de trabajo y justificar la contratación de nuevos Ingenieros o Analistas.

In [ ]:
query_6 = '''
    SELECT e.nombre AS Equipo_Gestor,
           COUNT(a.id_activo) AS Volumen_Objetos_Asignados
    FROM equipo e
    JOIN equipo_trabaja_objeto eto ON e.id_equipo = eto.id_equipo
    JOIN activo a ON eto.id_activo = a.id_activo
    GROUP BY e.nombre
'''
resultado_6 = spark.sql(query_6)

# Mostrar los resultados como tablas
resultado_6.show(truncate=False)

**Análisis:** Detectamos un desbalance en la asignación de responsabilidades. Los equipos de negocio (Contabilidad y Analítica) mantienen el doble de activos que el equipo central de "Datos e Inteligencia", indicando la necesidad de redistribuir la propiedad de los activos (Data Ownership).